# Week 2 - Day 1 - Hands On Lab


## Introduction

**Dataset:** Medical Cost Personal Dataset (Kaggle)

**Source:** `data/insurance.csv`

**Size:** 1,338 records, 7 columns


This dataset contains medical insurance billing information for 
individuals in the US.

Before describing the columns, I'll load the data first and check the real column names and types 
instead of guessing.

In [63]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("data/insurance.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

Shape: (1338, 7)

Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

Data types:
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object

First 5 rows:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


Based on the actual columns and types above, here's what each represents:

| Column | Type | Description |
|---|---|---|
| `age` | int | Age of the primary beneficiary (18-64 in this data) |
| `sex` | object | Gender (`male` / `female`) |
| `bmi` | float | Body Mass Index |
| `children` | int | Number of dependents covered by insurance (0-5) |
| `smoker` | object | Whether the person smokes (`yes` / `no`) |
| `region` | object | Residential area in the US |
| `charges` | float | Individual medical costs billed by insurance (the column I'll focus this analysis on) |

**Goal of this analysis:** Apply today's descriptive statistics 
(mean, median, mode, standard deviation, IQR) to `charges` - the most 
analytically interesting numeric column, since medical costs are known 
to vary widely and are often skewed by a small number of high-cost cases.

### Quick Data Check

Before trusting any statistics I compute from this data, I ran a quick 
check to confirm it's clean and ready for analysis.

In [64]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Found 1 duplicate row => removing it so it doesn't slightly bias our statistics
df = df.drop_duplicates()
print("\nShape after removing the duplicate:", df.shape)

Missing values per column:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicate rows: 1

Shape after removing the duplicate: (1337, 7)


## Step 1
Load a numeric column from the dataset into a Pandas Series.

I select `charges` specifically, since it's the target variable most 
relevant for statistical analysis - and, as I'll show below, it tells an 
interesting story about the spread of medical costs.

In [65]:
charges = df["charges"]
print(charges.describe())

count     1337.000000
mean     13279.121487
std      12110.359656
min       1121.873900
25%       4746.344000
50%       9386.161300
75%      16657.717450
max      63770.428010
Name: charges, dtype: float64


## Step 2
Compute mean, median, mode, standard deviation, and IQR for `charges`.

In [66]:
mean_val = np.mean(charges)
median_val = np.median(charges)
mode_result = stats.mode(charges, keepdims=False)
std_val = np.std(charges)
q1, q3 = np.percentile(charges, [25, 75])
iqr = q3 - q1

print("Mean:", round(mean_val, 2))
print("Median:", round(median_val, 2))
print("Mode:", mode_result.mode, "(appears", mode_result.count, "times)")
print("Standard Deviation:", round(std_val, 2))
print("IQR:", round(iqr, 2))

Mean: 13279.12
Median: 9386.16
Mode: 1121.8739 (appears 1 times)
Standard Deviation: 12105.83
IQR: 11911.37


## Step 3

**Justification:** The mean is about $3,900 higher than the median. 
Since the data isn't symmetric, this makes sense - a few patients with 
very high medical bills are pulling the average up, so the median 
represents a "typical" patient better than the mean does.

The mode isn't really useful here since `charges` is continuous - the 
most repeated value only shows up twice out of 1,337 rows, so it doesn't 
mean much. Mode makes more sense for a column like `smoker` or `region`.

In [67]:
gap = mean_val - median_val
print("Gap between mean and median:", round(gap, 2))
print("This confirms right-skewed data - a long tail of high-cost cases.")

Gap between mean and median: 3892.96
This confirms right-skewed data - a long tail of high-cost cases.


## Step 4 - Summary

The typical patient pays around **$9,386** (median), not the mean 
($13,279) - the mean is misleading here because of the skew. The spread 
is large too: std is about $12,106, and the middle 50% of patients (IQR) 
range from about $4,740 to $16,640.

This makes me want to check later in the week whether smoking status or 
age is connected to these high charges - that's something I can look at 
once we get to EDA.